# Oracle SQL Performance Benchmark Analysis

Load results from `bench_results.json` and compare variant performance.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

with open('bench_results.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f'{len(df)} records, {df.experiment.nunique()} experiments')
df.head()

In [ ]:
# Summary statistics per experiment + variant
summary = df.groupby(['experiment', 'variant'])['elapsed_sec'].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).round(4)
summary

In [ ]:
# Speedup ratios: slowest / fastest variant per experiment
medians = df.groupby(['experiment', 'variant'])['elapsed_sec'].median()
speedups = []
for exp in df.experiment.unique():
    vals = medians[exp]
    slow_name = vals.idxmax()
    fast_name = vals.idxmin()
    ratio = vals[slow_name] / vals[fast_name] if vals[fast_name] > 0 else float('inf')
    speedups.append({
        'experiment': exp,
        'slow_variant': slow_name,
        'fast_variant': fast_name,
        'slow_median_sec': round(vals[slow_name], 4),
        'fast_median_sec': round(vals[fast_name], 4),
        'speedup': f'{ratio:.1f}x',
    })

speedup_df = pd.DataFrame(speedups)
speedup_df

In [ ]:
# Bar charts: elapsed time per variant within each experiment
experiments = df.experiment.unique()
n = len(experiments)
cols = 2
rows = (n + 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, 3.5 * rows))
axes = axes.flatten()

for i, exp in enumerate(experiments):
    ax = axes[i]
    sub = df[df.experiment == exp]
    means = sub.groupby('variant')['elapsed_sec'].mean()
    stds = sub.groupby('variant')['elapsed_sec'].std()
    means.plot.bar(ax=ax, yerr=stds, capsize=4, color=['#e74c3c', '#2ecc71'][:len(means)])
    ax.set_title(exp)
    ax.set_ylabel('elapsed (sec)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('bench_elapsed.png', dpi=120)
plt.show()

In [ ]:
# Row count comparison (sanity check — variants should return same rows)
row_summary = df.groupby(['experiment', 'variant'])['rows'].first()
row_summary

In [ ]:
# Statistical significance: Mann-Whitney U test between slow/fast variants
from scipy.stats import mannwhitneyu

sig_results = []
for exp in df.experiment.unique():
    variants = df[df.experiment == exp].variant.unique()
    if len(variants) != 2:
        continue
    a = df[(df.experiment == exp) & (df.variant == variants[0])]['elapsed_sec']
    b = df[(df.experiment == exp) & (df.variant == variants[1])]['elapsed_sec']
    if len(a) < 3 or len(b) < 3:
        continue
    stat, pval = mannwhitneyu(a, b, alternative='two-sided')
    sig_results.append({
        'experiment': exp,
        'variant_a': variants[0],
        'variant_b': variants[1],
        'U_statistic': stat,
        'p_value': round(pval, 6),
        'significant': 'Yes' if pval < 0.05 else 'No',
    })

pd.DataFrame(sig_results)

In [ ]:
# Export summary as markdown table
lines = ['| Optimization | Slow Variant (sec) | Fast Variant (sec) | Speedup |',
         '|---|---|---|---|']
for _, row in speedup_df.iterrows():
    lines.append(f"| {row['experiment']} | {row['slow_median_sec']} | {row['fast_median_sec']} | {row['speedup']} |")

md = '\n'.join(lines)
print(md)

with open('bench_summary.md', 'w') as f:
    f.write(md + '\n')